# 04 · Research prompts, L1 and L2 (Day 4)
- System prompt = annotation rules (thesis §3.4) + role (F3 §3.7.4.1) + JSON schema.
- L1 zero-shot · L2 fixed few-shot (6 train examples, one per sarcasm × sentiment cell, chosen **by rule, not by hand**).
- Batch runner: resume, budget cap, test guard.

Done when (Dev Plan §8, Day 4): L1 and L2 run on the 20 development comments; parse rate logged.
Scores on 20 comments are **sanity checks only** — no design decision is made from them.

In [1]:
import sys, json, datetime
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from sarcllm.config import load_base, load_condition
from sarcllm.llm.client import LLMClient, LLMSettings
from sarcllm.prompts import Prompts, item_view
from sarcllm.pipelines.single_pass import run_item
from sarcllm.runner import run, load_predictions, FrozenGuardError
from sarcllm.eval.metrics import align, compute

base = load_base(ROOT)
client = LLMClient(LLMSettings.from_env(ROOT / ".env"), cache_path=ROOT / base["paths"]["cache"])
prompts = Prompts(ROOT / base["paths"]["prompts"])
TF, CF = base["input"]["text_field"], base["input"]["context_field"]
print("generation :", base["generation"])
print("input      :", TF, "/", CF, "  · frozen:", base["frozen"])

generation : {'temperature': 0.0, 'max_tokens': 2048, 'reasoning_effort': 'low', 'json_mode': True}
input      : text_enc / context_enc   · frozen: False


In [2]:
def load(split):
    df = pd.read_json(ROOT / "data" / "processed" / f"{split}.jsonl", lines=True, dtype=False)
    for c in ("id", "post_id"):
        df[c] = df[c].astype(str)
    return df

train, val = load("train"), load("val")
assert {TF, CF} <= set(val.columns), "run notebook 01b first (text_enc / context_enc missing)"
dev_ids = (ROOT / "data" / "eval_sets" / "dev_val_20.txt").read_text(encoding="utf-8").split()
dev = val.set_index("id").loc[dev_ids].reset_index()
print(f"train {len(train):,} · val {len(val):,} · dev {len(dev)}")

train 32,582 · val 6,938 · dev 20


## Few-shot set for L2 (written once, then committed)
**Rule** (no hand-picking, so no researcher choice of "good" examples): from **train** only, one example
per sarcasm × sentiment cell; each cell assigned one script so the six cover 2 × Sinhala, 2 × Latin,
2 × mixed; comment 15–200 characters; caption non-empty and ≤ 300 characters (keeps the prompt short);
one row drawn at random with seed 42. If a cell has no candidate, the script filter and then the length
filter are dropped for that cell (recorded in `notes`).

In [4]:
FS_PATH = ROOT / "prompts" / "few_shot.json"
CELL_SCRIPT = {(1, "negative"): "sinhala", (0, "negative"): "latin",
               (1, "neutral"):  "latin",   (0, "neutral"):  "mixed",
               (1, "positive"): "mixed",   (0, "positive"): "sinhala"}

if not FS_PATH.exists():
    examples, notes = [], []
    ok_len = train[TF].str.len().between(15, 200) & train[CF].str.len().between(1, 300)
    for (sar, sent), script in CELL_SCRIPT.items():
        cell_all = train[(train.is_sarcastic == sar) & (train.sentiment == sent)]
        cell = cell_all[ok_len.loc[cell_all.index]]
        cand = cell[cell.script_type == script]
        if cand.empty:
            cand = cell
            notes.append(f"cell ({sar},{sent}): no {script} candidate, script filter dropped")
        if cand.empty:
            cand = cell_all
            notes.append(f"cell ({sar},{sent}): no candidate within length limits, length filter dropped")
        row = cand.sample(1, random_state=42).iloc[0]
        examples.append(item_view(row, TF, CF))
    FS_PATH.write_text(json.dumps({
        "rule": "train only; one per sarcasm x sentiment cell; script per cell as CELL_SCRIPT; "
                "comment 15-200 chars; caption 1-300 chars; random_state=42",
        "cell_script": {f"{k[0]}_{k[1]}": v for k, v in CELL_SCRIPT.items()},
        "notes": notes, "examples": examples}, ensure_ascii=False, indent=2), encoding="utf-8")
    print("wrote", FS_PATH.relative_to(ROOT), "·", notes or "no fallbacks")

few_shot = prompts.few_shot()
display(pd.DataFrame(few_shot)[["id", "is_sarcastic", "sentiment", "script_label", "text", "context"]])

wrote prompts\few_shot.json · no fallbacks


,id,is_sarcastic,sentiment,script_label,text,context
0,P545C40,1,negative,Sinhala script,"ආයෙ ඉතින්😏...""උසනම් සිකුරිටි🤌මිටි නම් කම්කරු""😂...",ස.ජ.බයේ බුද්ධිමට්ටම අනෙක් පක්ෂවලට ලංවීමටවත් නො...
1,P358C57,0,negative,Latin script (Singlish or English),Nikan labunanam sadda nathuwa kala hitapan mad...,Mr Kottu එකේ රයිස් එක ඇත්තටම රස ?
2,P742C43,1,neutral,Latin script (Singlish or English),Adeh kukku dileka neda adala thiyenne 😂😂,නූගත් ගැහැණුන්ට Tiktok එකක් තියනව කියන්නෙ බඩුම...
3,P39C71,0,neutral,mixed Sinhala and Latin script,ගෙවල් ඉබේ hadhenawada,ඇත්තටම ගෙයක rent එක කියන්නේ කිසිම වියදමක් නැතු...
4,P532C23,1,positive,mixed Sinhala and Latin script,"STF,එකටත් කාලෙකට පස්සෙ ගිනිඅවි පුහුණුවකට (හරඹය...",මීගමුව නිකන් නන්දිකඩාල් වගේ. තියපු වෙඩි වල හැට...
5,P755C12,0,positive,Sinhala script,උඹල වීඩියෝ කර කර කියන්න දෙයක් නැහැ බං.කොහොමත් ...,Siriwardana food cafe වැඩිපුර රස මොකක්ද? රයිස්...


In [5]:
val_test_posts = set(val.post_id) | set(load("test").post_id)
fs_rows = train.set_index("id").loc[[e["id"] for e in few_shot]]
assert len(fs_rows) == 6
assert not set(fs_rows.post_id) & val_test_posts, "few-shot example shares a post with val/test"
assert not {e["id"] for e in few_shot} & set(dev.id)
print("Few-shot leakage checks: PASS (all 6 from train, no shared posts with val/test)")
prompts = Prompts(ROOT / base["paths"]["prompts"])   # reload so version() includes few_shot.json
print("prompt version:", prompts.version())

Few-shot leakage checks: PASS (all 6 from train, no shared posts with val/test)
prompt version: cf32f40f5860


In [6]:
item = item_view(dev.iloc[0], TF, CF)
l2 = prompts.single_pass(item, few_shot)
print("=== SYSTEM ===\n" + l2[0]["content"])
print("\n=== USER (L2) ===\n" + l2[1]["content"])

=== SYSTEM ===
You are a Sinhala linguistics expert who annotates Sri Lankan social-media comments. You read Sinhala script, Singlish (Sinhala written in Latin script), English, and Sinhala–English code-mixed text. You understand informal writing conventions: emoji, internet slang, abbreviations, exaggerated expressions, rhetorical questions and culturally specific references.

Each item is a COMMENT written in reply to a social-media POST CAPTION (the context). The caption may be cut off. You label the comment, using the caption as context. Do not translate the text.

LABEL DEFINITIONS (corpus annotation guideline)
- Sarcastic: the comment's communicative intent contradicts its literal surface sentiment, exploiting a gap between what is said and the situational or cultural context.
- Non-sarcastic: the literal surface sentiment corresponds to the communicative intent.
- Literal (surface) sentiment: the polarity a literal, context-free reading of the comment would suggest.
- Intended (

In [7]:
async def run_condition(cond_id, rows, set_name, split="val"):
    cond = load_condition(ROOT, cond_id)
    examples = few_shot if cond["examples"] == "few_shot" else []
    fn = lambda r: run_item(client, prompts, item_view(r, TF, CF), condition=cond["id"],
                            gen=base["generation"], examples=examples)
    return await run(cond=cond, base=base, rows=rows, split=split, set_name=set_name, client=client,
                     item_fn=fn, root=ROOT, prompt_version=prompts.version(),
                     extra_meta={"example_ids": [e["id"] for e in examples]})

preds = {}
preds["L1"] = await run_condition("L1", dev, "dev20")

[l1_val_dev20] 0 already done · 20 to run
  20/20 complete · 0 errors
[l1_val_dev20] this session: 20 done, 0 errors · total predictions 20/20


In [8]:
preds["L2"] = await run_condition("L2", dev, "dev20")

[l2_val_dev20] 0 already done · 20 to run
  20/20 complete · 0 errors
[l2_val_dev20] this session: 20 done, 0 errors · total predictions 20/20


In [9]:
rows = []
for c, p in preds.items():
    a = align(dev, p[["id", "sentiment_pred", "is_sarcastic_pred"]], dev.id.tolist())
    m = compute(a)
    rows.append({"condition": c, "n": len(p), "parse_ok": p.parse_ok.mean(), "repaired": (p.n_calls > 1).sum(),
                 "evidence_valid": p.evidence_valid.mean(), "finish_length": (p.finish_reason == "length").sum(),
                 "tokens_in/item": p.tokens_in.mean(), "tokens_out/item": p.tokens_out.mean(),
                 "usd/item": p.cost_usd.mean(), "sent_f1": m["sent_f1_overall"],
                 "sarc_subset_f1": m["sent_f1_sarcastic"], "det_f1": m["det_f1_macro"]})
check = pd.DataFrame(rows).set_index("condition")
display(check.round(3))
print("n = 20: these F1 values are sanity checks, not results.")

,n,parse_ok,repaired,evidence_valid,finish_length,tokens_in/item,tokens_out/item,usd/item,sent_f1,sarc_subset_f1,det_f1
condition,,,,,,,,,,,
L1,20,1.0,0,1.0,0,754.05,191.45,0.0,0.571,0.349,0.286
L2,20,1.0,0,1.0,0,1438.05,200.70,0.0,0.571,0.238,0.405


n = 20: these F1 values are sanity checks, not results.


In [10]:
side = dev[["id", "script_type", "is_sarcastic", "sentiment", TF]].copy()
for c, p in preds.items():
    q = p.set_index("id")
    side[f"{c}_sarc"] = side.id.map(q.is_sarcastic_pred)
    side[f"{c}_sent"] = side.id.map(q.sentiment_pred)
display(side)
print("\nL2 rationales (first 5):")
for r in preds["L2"].head(5).itertuples():
    print(f"- {r.id}: {r.rationale}")

,id,script_type,is_sarcastic,sentiment,text_enc,L1_sarc,L1_sent,L2_sarc,L2_sent
0,P125C80,sinhala,0,positive,සිතුවිල්ල දහස් වටි,0,positive,0,positive
1,P137C43,mixed,1,negative,v8 මදී වගේ අපි උත්සාහ කරමු හේලිකෝප්ටර් එකක් දේන්න,0,positive,0,positive
2,P166C94,latin,0,positive,Theruwan saraayi!,0,positive,0,positive
3,P175C53,sinhala,0,negative,මොක්කු සහෝදරියෝ යකො පින්නයෝ කියලා දැම්මනම් හරි,0,negative,0,negative
4,P175C71,latin,1,negative,Pissu ponnya 😂,0,negative,0,negative
5,P239C46,mixed,0,negative,ඒ උනාට බං උඹ හිතන්නකො උන් ආපු වේගෙට අහින්සක මන...,0,negative,0,negative
6,P247C16,sinhala,1,negative,"ලංගම මෙට්රෝ බස් නිසා ගොඩ යනවනම් අපි චුන් වෙමු,...",0,negative,0,negative
7,P277C3,mixed,0,positive,අඩෝ නියමයි Madubashana prabath ranahansha අයියේ❤️,0,positive,0,positive
8,P325C25,sinhala,1,negative,අනුර ප්‍රතිපත්තියක් නැතුව ලැජ්ජා වෙනවා ලැජ්ජා ...,0,negative,0,negative
9,P325C72,mixed,1,negative,තක්කඩියා අවසන් නවාතැන hora,0,negative,0,negative



L2 rationales (first 5):
- P175C53: The comment uses vulgar insult 'පින්නයෝ', giving a clearly negative surface tone. No reversal or ironic contrast with the affectionate caption is evident, so it is not sarcastic.
- P175C71: The comment uses insulting terms 'pissu ponnya' (crazy idiot) indicating a negative sentiment; no irony or contrast with the affectionate caption, so it's non‑sarcastic.
- P125C80: The comment literally praises the picture as 'worth thousands', a positive statement. No sarcastic cues or contrast with the political caption are evident, so it is taken at face value.
- P137C43: The comment uses upbeat, encouraging wording ('let's try', 'give a helicopter') without irony; given the caption about a V8, it reads as a playful positive suggestion, not sarcasm.
- P277C3: The comment uses enthusiastic phrase 'අඩෝ නියමයි' and a heart emoji, expressing approval. No ironic contrast with the caption, so it's sincere positive sentiment.


In [11]:
fails = pd.concat([p[~p.parse_ok] for p in preds.values()])
display(fails[["id", "condition", "parse_error", "finish_reason"]] if len(fails) else "no parse failures")

'no parse failures'

In [12]:
test_rows = load("test").head(2)
try:
    await run_condition("L1", test_rows, "guardcheck", split="test")
    raise AssertionError("test run was NOT blocked")
except FrozenGuardError as e:
    print("Blocked as intended:", e)

Blocked as intended: split='test' is blocked: set frozen: true in configs/base.yaml at Gate G1


In [13]:
u = client.usage()["all_time"]
print(f"""## {datetime.date.today()} · Prompts, L1, L2 (notebook 04)
- Prompt version {prompts.version()} (prompts/*.j2 + few_shot.json). System prompt: thesis §3.4 label definitions and
  decision order (surface → sarcasm → intended); F3 §3.7.4 role; JSON schema. Input fields {TF}/{CF}.
- Few-shot (L2): {[e['id'] for e in few_shot]} — drawn by rule (train, 1 per cell, seed 42), not hand-picked.
  Leakage checks pass.
- Dev run (20 val comments): parse rate L1 {check.loc['L1','parse_ok']:.0%}, L2 {check.loc['L2','parse_ok']:.0%};
  repaired L1 {check.loc['L1','repaired']}, L2 {check.loc['L2','repaired']}; tokens in/item L1 {check.loc['L1','tokens_in/item']:.0f}, L2 {check.loc['L2','tokens_in/item']:.0f}.
- Test guard verified (split=test refused while frozen: false).
- Cumulative usage: {u['requests']} paid requests, ${u['cost_usd']:.4f}.
""")

## 2026-11-21 · Prompts, L1, L2 (notebook 04)
- Prompt version cf32f40f5860 (prompts/*.j2 + few_shot.json). System prompt: thesis §3.4 label definitions and
  decision order (surface → sarcasm → intended); F3 §3.7.4 role; JSON schema. Input fields text_enc/context_enc.
- Few-shot (L2): ['P545C40', 'P358C57', 'P742C43', 'P39C71', 'P532C23', 'P755C12'] — drawn by rule (train, 1 per cell, seed 42), not hand-picked.
  Leakage checks pass.
- Dev run (20 val comments): parse rate L1 100%, L2 100%;
  repaired L1 0, L2 0; tokens in/item L1 754, L2 1438.
- Test guard verified (split=test refused while frozen: false).
- Cumulative usage: 79 paid requests, $0.0151.

